# Country inputs: boundaries, population, buildings and GDP

Scratch notebook, built together. For one country, year, resolution and building type it loads:

1. the admin-0 polygons (simplified) of every country in that country's GBD region;
2. the admin-2 polygons (simplified) of the country;
3. the population raster for the year;
4. the building rasters of the period closest to the year;
5. a GDP per capita raster: each pixel takes the value of the admin-2 unit it falls in;

and draws the population raster with admin-2 lines (thin) and admin-0 lines (bold) on top.

It then holds all the layers together: one Dataset on the pixel grid, and one table with a row per pixel inside the country.

Every function is imported from `idd_human_suitability.lib` or a sibling package. Autoreload is on, so an edit to a `lib` function takes effect on the next cell run without restarting the kernel.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
from idd_spatial_analysis.boundaries import load_admin
from idd_spatial_analysis.constants import ADM_CODE_COLUMN, LOC_ID_COLUMN, LOCATION_ID_COLUMN
from idd_spatial_analysis.raster import rasterize_zones

from idd_human_suitability import constants as c
from idd_human_suitability.lib.admin2_income import load_admin2_gdppc
from idd_human_suitability.lib.boundaries import (
    admin_release,
    country_admin2,
    find_country,
    load_gbd_hierarchy,
    region_admin0,
)
from idd_human_suitability.lib.buildings import (
    available_bands,
    available_periods,
    blocks_for,
    building_raster,
    closest_period,
)
from idd_human_suitability.lib.figures.maps import raster_map_with_admin_lines
from idd_human_suitability.lib.pixels import pixels_to_table, stack_layers
from idd_human_suitability.lib.population import population_window
from idd_human_suitability.lib.rasterize import zone_values_to_raster

## Parameters

The first four are the inputs. The rest are the releases in use, read from `constants.py`; assign a different value here to try another.

In [ ]:
COUNTRY = "Kenya"
YEAR = 2020
RESOLUTION = 100  # metres; 100 is the only resolution in use for now
BUILDING_TYPE = "microsoft"  # "ghsl" or "microsoft"

ADMIN_RELEASE = c.ADMIN_RELEASE
GBD_HIERARCHY = c.GBD_HIERARCHY
POPULATION_VERSION = c.POPULATION_VERSION
BUILDING_SOURCE = c.BUILDING_SOURCES[BUILDING_TYPE]
GDP_SCENARIO = "reference"  # the GDP table also has "better" and "worse"

print(f"{COUNTRY}, {YEAR}, {RESOLUTION} m, buildings from {BUILDING_SOURCE}")
print(f"admin release {ADMIN_RELEASE}, hierarchy {GBD_HIERARCHY}, population {POPULATION_VERSION}")

## 1. Admin-0 polygons of the country's region

The shapefiles have no grouping above the country, so the region comes from the GBD hierarchy: the shapefile's `loc_id` is the hierarchy's `location_id`.

In [ ]:
release = admin_release(ADMIN_RELEASE)
admin0_gdf = load_admin(release, 0, columns=[LOC_ID_COLUMN])
hierarchy_df = load_gbd_hierarchy(GBD_HIERARCHY)

country = find_country(COUNTRY, admin0_gdf, hierarchy_df)
region_gdf = region_admin0(country, admin0_gdf, hierarchy_df)

print(country)
print(f"{len(region_gdf)} countries in {country.region_name}")
region_gdf[["ADM0_CODE", "ADM0_NAME", LOC_ID_COLUMN]]

## 2. Admin-2 polygons of the country

In [ ]:
admin2_gdf = country_admin2(country, release)

print(f"{len(admin2_gdf)} admin-2 units; bounding box (lon/lat) {admin2_gdf.total_bounds.round(3)}")
admin2_gdf.head()

## 3. Population raster

One window of the global raster, covering the admin-2 bounding box. It is on the population model's equal-area grid, and it includes the pixels of neighbouring countries that fall inside the box.

In [ ]:
bounds = tuple(float(value) for value in admin2_gdf.total_bounds)
population_arr, grid = population_window(
    bounds, YEAR, resolution=RESOLUTION, version=POPULATION_VERSION
)

print(f"shape {population_arr.shape}, {population_arr.nbytes / 1e6:.0f} MB, {population_arr.dtype}")
print(f"grid: {grid.crs}, pixel {grid.resolution} m, bounds {tuple(round(v) for v in grid.bounds)}")
print(f"people in the window: {np.nansum(population_arr):,.0f}")

## 4. Building rasters for the closest period

Periods are named `<year>q<quarter>`. The closest one is measured from the first quarter of the year, which is when the population rasters are dated. Every band that exists for all of the country's tile blocks is loaded, on the same grid as the population raster.

In [ ]:
periods = available_periods(BUILDING_SOURCE, RESOLUTION)
period = closest_period(YEAR, periods)
blocks = blocks_for(admin2_gdf, RESOLUTION)
bands = available_bands(BUILDING_SOURCE, period, blocks, resolution=RESOLUTION)

building_arrs = {
    band: building_raster(BUILDING_SOURCE, period, band, blocks, grid, resolution=RESOLUTION)
    for band in bands
}

print(
    f"periods on disk: {periods[0]} to {periods[-1]} ({len(periods)}); closest to {YEAR}: {period}"
)
print(f"tile blocks: {blocks}")
for band, building_arr in building_arrs.items():
    print(
        f"{band}: shape {building_arr.shape}, mean {np.nanmean(building_arr):.5f}, "
        f"max {np.nanmax(building_arr):.3f}, share without data {np.isnan(building_arr).mean():.3f}"
    )

## 5. GDP per capita raster

The admin-2 GDP per capita values for the country, year and scenario, put on the raster grid: the admin-2 polygons are burned to a zone mask (`loc_id` per pixel) and each pixel takes its zone's value. Pixels outside the country are NaN.

In [ ]:
gdppc_df = load_admin2_gdppc(country.iso3, YEAR, GDP_SCENARIO)
zone_mask, _ = rasterize_zones(admin2_gdf, LOC_ID_COLUMN, grid)
gdppc_arr = zone_values_to_raster(zone_mask, gdppc_df.set_index(LOCATION_ID_COLUMN)[c.GDPPC_COLUMN])

print(
    f"{len(gdppc_df)} admin-2 units with GDP per capita; {np.unique(zone_mask).size - 1} got pixels"
)
print(
    f"raster: shape {gdppc_arr.shape}, min {np.nanmin(gdppc_arr):,.1f}, max {np.nanmax(gdppc_arr):,.1f}"
)
print(
    f"people inside {country.name} (pixels in an admin-2 unit): {np.nansum(population_arr[zone_mask > 0]):,.0f}"
)
gdppc_df.head()

## 6. Map: population with admin-2 and admin-0 lines

Drawn with idd-figures through `lib/figures/maps.py`. The extent is the admin-2 bounding box. For drawing, the raster is averaged onto a lon/lat grid of `DISPLAY_RESOLUTION` degrees, so the colours are people per source pixel, averaged over each display cell.

In [ ]:
POPULATION_BINS = [0, 0.01, 0.1, 1, 5, 25, 100, 1000]
DISPLAY_RESOLUTION = 0.01  # degrees

fig = raster_map_with_admin_lines(
    population_arr,
    grid,
    admin0_gdf=region_gdf,
    admin2_gdf=admin2_gdf,
    bins=POPULATION_BINS,
    display_resolution=DISPLAY_RESOLUTION,
    resampling="average",
    legend_title=f"People per {RESOLUTION} m pixel (mean over {DISPLAY_RESOLUTION}° cell), {YEAR}",
    title=f"{country.name}: population, {YEAR}",
)

## 7. All layers in one Dataset

Every array above is on the same grid, so they stack into one `xarray.Dataset` with dimensions `(y, x)`: one variable per layer, with the pixel centres as coordinates (in the grid's equal-area CRS, metres). The arrays are wrapped, not copied. `admin2_loc_id` is the LSAE location id the GDP table joins on; `admin2_code` is the shapefile's `ADM2_CODE`. Both are 0 outside the country.

In [ ]:
admin2_code_mask, _ = rasterize_zones(admin2_gdf, ADM_CODE_COLUMN.format(level=2), grid)

pixels_ds = stack_layers(
    {
        "population": population_arr,
        **{f"building_{band}": building_arr for band, building_arr in building_arrs.items()},
        "gdppc": gdppc_arr,
        "admin2_loc_id": zone_mask,
        "admin2_code": admin2_code_mask,
    },
    grid,
)

print(f"{pixels_ds.nbytes / 1e9:.1f} GB in {len(pixels_ds.data_vars)} variables")
pixels_ds

## 8. One row per pixel inside the country

The same layers as a table: one row for each pixel that falls in an admin-2 unit of the country, with its position and every layer's value.

In [ ]:
pixels_df = pixels_to_table(pixels_ds, zone_variable="admin2_loc_id")

print(
    f"{len(pixels_df):,} pixels inside {country.name}; {pixels_df.memory_usage().sum() / 1e9:.1f} GB"
)
print(f"people: {pixels_df['population'].astype('float64').sum():,.0f}")
pixels_df.head()

In [ ]:
# One pixel, all of its values: here the pixel with the most people.
pixels_df.loc[[pixels_df["population"].idxmax()]]